In [1]:
import re
import yaml
import polars as pl
from pathlib import Path
from functools import reduce
from typing import List, Union

def aggregate_run_data(search_paths: Union[str, Path, List[Union[str, Path]]], properties: List[str]) -> pl.DataFrame:
    paths = [Path(search_paths)] if isinstance(search_paths, (str, Path)) else [Path(p) for p in search_paths]
    get_prop = lambda d, p: reduce(lambda c, k: c.get(k) if isinstance(c, dict) else None, p.split('.'), d)
    
    def process_dir(cfg: Path) -> dict:
        try: d = yaml.safe_load(cfg.read_text()) or {}
        except Exception: d = {}
        
        ckpts = [(int(m.group(1)), int(m.group(2))) for f in cfg.parent.glob("checkpoint_*.pt") 
                 if (m := re.search(r"epoch_(\d+)_model_(\d+)\.pt", f.name))]
        eps, mods = ([c[0] for c in ckpts], [c[1] for c in ckpts]) if ckpts else ([], [])
        
        return {
            "run_dir": str(cfg.parent),
            **{p: get_prop(d, p) for p in properties},
            "num_models": max(mods) if mods else None,
            "max_epoch": max(eps) if eps else None,
            "epoch_range": sorted(set(eps)) if eps else [] # List of all unique epochs
        }

    # Flattened comprehension now strictly ignores any path containing a '.hydra' folder
    return pl.DataFrame([
        process_dir(cfg) 
        for p in paths 
        for cfg in p.rglob("config.yaml") 
        if ".hydra" not in cfg.parts
    ])


In [ ]:
# Example usage
df = aggregate_run_data(
    search_paths=["outputs/"], # Specify more specific sub directories if necessary
    properties=[
        "model.symmetry",
        "model.mask_params.linear_mask_params_0.mask_constant"
    ]
).sort("run_dir")
df.show(limit=None)
{
    f"mlp_symmetry{row["model.symmetry"]}_kappa{int(row["model.mask_params.linear_mask_params_0.mask_constant"])}": row["run_dir"]
    for row in df.iter_rows(named=True)
}